# Module 2: Chart generation

In [2]:
#Library imports
import re
import json
import utils

In [4]:
#load dataset
# Use this utils.py function to load the data into a dataframe
df = utils.load_and_prepare_data('data/coffee_sales.csv')

# Grab a random sample to display
utils.print_html(df.sample(n=5), title="Random Sample of Coffee Sales Data")

date,time,cash_type,card,price,coffee_name,quarter,month,year
2025-02-18,12:27,card,ANON-0000-0000-1225,3.086,Americano with Milk,1,2,2025
2024-03-09,13:36,card,ANON-0000-0000-0043,3.870,Cappuccino,1,3,2024
2025-02-03,07:01,card,ANON-0000-0000-1152,3.086,Americano with Milk,1,2,2025
2024-03-22,13:48,card,ANON-0000-0000-0012,2.400,Espresso,1,3,2024
2024-10-29,15:31,card,ANON-0000-0000-0836,3.086,Americano with Milk,4,10,2024


Goal: build a data analytics pipeline
Pipeline steps
1. Prompt -> LLM -> Version 1 (V1) code
2. Execute LLM code -> output chart
3. feed chart into LLM to critique and update code -> V2 code
4. Execute code -> output chart

In [ ]:
#Step 1: Prompt -> LLM -> V1 code

def generate_chart_code(instruction: str, model: str, out_path_v1: str) -> str:
    """generate Python code to make a plot with matplotlib using tag-based wrapping"""
    
    prompt = f"""
    You are a data visualization expert.

    Return your answer *strictly* in this format:

    <execute_python>
    #valid python code here
    </execute_python>

    Do not add explanations, only the tags and the code.

    The code should create a visualization from a DataFrame 'df' with these columns:
    - date (datetime64 - already parsed; use df['date'].dt.year, df['date'].dt.month, etc.)
    - time (string, HH:MM - do NOT concatenate or combine with the date column)
    - cash_type (string: 'card' or 'cash')
    - card (string)
    - price (number)
    - coffee_name (string)
    - quarter (int, 1-4 - already computed, use directly)
    - month (int, 1-12 - already computed, use directly)
    - year (int, e.g. 2024 - already computed, use directly)

    User Instruction: {instruction}

    Requirements for the code:
    1. Assume the DataFrame is already loaded as 'df'
    2. Use matplotlib for plotting
    3. Add clear title, axis labels, and legend if needed
    4. Save the figure as '{out_path_v1}' with dpi=300.
    5. Do not call plt.show()
    6. Close all plots with plt.close()
    7. Add all necessary import python statements
    8. CRITICAL: 'date' is datetime64 - never use string concatenation on it.
    Filter by year/quarter using the 'year' and 'quarter' integer columns.

    Return ONLY the code wrapped in <execute_python> tags.
    """

    response = utils.get_response(model, prompt) #this abstracts the openAI SDK chat.completions api and parses response directly
    return response

#call the function to generate initial code and print it out for inspection
instruction = "Create a plot comparing Q3 coffee sales in 2024 and 2025 using data in coffee_sales.csv"
#here involves some prompt engineering~

code_v1 = generate_chart_code(
    instruction=instruction,
    model="gpt-4o-mini",
    out_path_v1="chart-v1.ipynb"
)

#again with utils help, print code in html
utils.print_html(code_v1, title="LLM output with first draft code")
#code wrapped inside <execute_python> tags

In [ ]:
#Step 2: execute code and create chart
# detailed steps:
# 1. extract code with regex
match = re.search(r"<xecute-python>([\s\S]*?)</execute_python>", code_v1)
# 2. execute code
if match:
    initial_code = match.group(1).strip()
    utils.print_html(initial_code, title="Extracted Code to Execute")
    # 3. generate chart
    exec_globals = {"df": df}
    exec(initial_code, exec_globals)

# 4. view chart in notebook
utils.print_html(
    content="chart_v1.png",
    title="Generated Chart (V1)",
    is_image=True
)

In [ ]:
#Step 3: Reflect on the input
def reflect_on_image_and_regenerate(
        chart_path: str,
        instruction: str,
        model_name: str,
        out_path_v2: str,
        code_v1: str,
) -> tuple[str, str]:
    #1) Docstring defining what this function does
    """
    Critique the chart IMAGE and the original code against the instruction,
    then return refined matplotlib code.
    Returns (feedback, refined_code_with_tags).
    Supports OpenAI and Anthropic (Claude)
    """
    #2) read the file from chart path with utils function (a wrapper function for base64)
    media_type, b64 = utils.encode_image_b64(chart_path)

    #3) define prompt for reflection, what this talks about:
    # a) Assistant role
    # b) one sentence define task
    # c) Context needed {}
    # d) Output format restrictions, list and picture in detail how the output should look like
    # e) Constraints in how the code should be designed, e.g. use what library/function, function input
    # f) Schema
    # g) instruction: {}
    prompt = f"""
    You are a data visualization expert.
    Your task: critique the attached chart and the original code against the given instruction,
    then return improved matplotlib code.

    Original code (for context):
    {code_v1}

    OUTPUT FORMAT (STRICT):
    1) First line: a valid JSON object with ONLY the "feedback" field.
    Example: {{"feedback": "The legend is unclear and the axis labels overlap."}}

    2) After a newline, output ONLY the refined Python code wrapped in:
    <execute_python>
    ...
    </execute_python>

    3) Import all necessary libraries in the code. Don't assume any imports from the original code.

    HARD CONSTRAINTS:
    - Do NOT include Markdown, backticks, or any extra prose outside the two parts above.
    - Use pandas/matplotlib only (no seaborn).
    - Assume df already exists; do not read from files.
    - Save to '{out_path_v2}' with dpi=300.
    - Always call plt.close() at the end (no plt.show()).
    - Include all necessary import statements.
    
    IMPORTANT: The 'date' column is already a pandas datetime64 type.
    - Do NOT concatenate 'date' with 'time' using string operations.
    - To filter by year/quarter, use: df[df['year'] == 2024] or df['date'].dt.year == 2024
    - The 'quarter' and 'year' columns already exist as integers; use them directly.

    Schema (columns available in df):
    - date   (datetime64 — already parsed; use df['date'].dt.year, etc.)
    - time   (string, HH:MM — do NOT concatenate with date)
    - cash_type (string: 'card' or 'cash')
    - card   (string)
    - price  (float)
    - coffee_name (string)
    - quarter (int, 1–4)
    - month  (int, 1–12)
    - year   (int)
    
    CRITICAL TYPE RULE: 'date' is already datetime64.
    - NEVER do: df['date'] + ' ' + df['time']  ← this will crash
    - ALWAYS filter by year/quarter using the integer columns: df[df['year'] == 2024]

    Instruction:
    {instruction}
    """

    #4) call utils function to generate image from LLM written code
    lower = model_name.lower()
    if "claude" in lower or "anthropic in lower":
        #the helper joins all text blocks and adds a system prompt
        content = utils.image_anthropic_call(model_name, prompt, media_type, b64)

    else:
        content = utils.image_openai_call(model_name, prompt, media_type, b64)

    #5) Parse and return output
    # a) parse only the first JSON line
    lines = content.strip().splitlines()
    json_line = lines[0].strip() if lines else ""

    try:
        obj = json.loads(json_line)
    except Exception as e:
        #fallback to capture first {} in all content
        m_json = 

    #5b) 

    